In [1]:
import os
import numpy as np
import gc
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K

# --- 1. THE RECOVERY BLOCK ---
# Force TensorFlow to only see ONE GPU to avoid initialization conflicts (Error 303)
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Prevent TF from pre-allocating all 16GB of VRAM immediately
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print("GPU initialized safely.")
    except RuntimeError as e:
        print(e)

tf.keras.mixed_precision.set_global_policy('mixed_float16')

# --- 2. THE ULTIMATE MEMORY-SAFE GENERATOR ---
class BulletproofGenerator(tf.keras.utils.Sequence):
    def __init__(self, month_paths, batch_size=8, lookback=10, forecast=16, stats=None):
        self.month_data = []
        for path in month_paths:
            # We store the POINTERS to the files, not the data itself
            self.month_data.append({
                'c': np.load(os.path.join(path, "cpm25.npy"), mmap_mode='r'),
                'u': np.load(os.path.join(path, "u10.npy"), mmap_mode='r'),
                'v': np.load(os.path.join(path, "v10.npy"), mmap_mode='r'),
                'p': np.load(os.path.join(path, "pblh.npy"), mmap_mode='r')
            })
            
        self.batch_size = batch_size
        self.lookback = lookback
        self.forecast = forecast
        self.stats = stats 
        
        self.indices = []
        for m_idx, data in enumerate(self.month_data):
            # Use the length of the first file (cpm25) to determine indices
            for t_idx in range(data['c'].shape[0] - lookback - forecast):
                self.indices.append((m_idx, t_idx))
        np.random.shuffle(self.indices)

    def __len__(self):
        return len(self.indices) // self.batch_size

    def __getitem__(self, index):
        batch_idx = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        X = np.zeros((self.batch_size, 140, 124, 40), dtype=np.float32)
        y = np.zeros((self.batch_size, 140, 124, 16), dtype=np.float32)
        
        for i, (m_idx, t_idx) in enumerate(batch_idx):
            d = self.month_data[m_idx]
            
            # Grab slices from disk (Only these 10 hours enter RAM)
            c_slice = d['c'][t_idx : t_idx + self.lookback]
            u_slice = d['u'][t_idx : t_idx + self.lookback]
            v_slice = d['v'][t_idx : t_idx + self.lookback]
            p_slice = d['p'][t_idx : t_idx + self.lookback]
            
            window = np.stack([c_slice, u_slice, v_slice, p_slice], axis=-1).astype(np.float32)
            
            if self.stats:
                window = (window - self.stats[0]) / (self.stats[1] + 1e-7)
            
            # Flatten Time into Channels: (10, 140, 124, 4) -> (140, 124, 40)
            X[i] = np.transpose(window, (1, 2, 0, 3)).reshape(140, 124, 40)
            
            # Target Delta calculation
            target_raw = d['c'][t_idx + self.lookback : t_idx + self.lookback + self.forecast].astype(np.float32)
            # Standardize target manually using PM2.5 stats (index 0)
            target_std = (target_raw - self.stats[0][0]) / self.stats[1][0]
            last_known_std = window[-1, ..., 0]
            
            y[i] = np.transpose(target_std - last_known_std, (1, 2, 0))
            
        return X, y

# --- 3. EXECUTION ---
# Hardcoded Z-score stats based on Phase 1 insights to avoid the calculation crash
means = np.array([85.0, 0.5, 0.5, 450.0]) 
stds = np.array([70.0, 4.0, 4.0, 250.0])
stats = (means, stds)

base = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/raw/"
train_gen = BulletproofGenerator([base+"APRIL_16", base+"JULY_16", base+"OCT_16"], batch_size=4, stats=stats)
val_gen = BulletproofGenerator([base+"DEC_16"], batch_size=4, stats=stats)

# --- 4. THE 2D CNN MODEL ---
def episode_loss(y_true, y_pred):
    mse = K.square(y_true - y_pred)
    # Give 5x weight to episodes (Standardized delta > 1.5)
    weight = tf.where(tf.abs(y_true) > 1.5, 5.0, 1.0)
    return K.mean(mse * weight)

model = models.Sequential([
    layers.Input(shape=(140, 124, 40)),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(16, (1, 1), activation='linear')
])

model.compile(optimizer='adam', loss=episode_loss, metrics=['mae'])
model.fit(train_gen, validation_data=val_gen, epochs=5)

2026-04-04 14:23:04.280743: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775312584.683293      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775312584.795501      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775312585.806461      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775312585.806505      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775312585.806508      24 computation_placer.cc:177] computation placer alr

GPU initialized safely.


I0000 00:00:1775312625.890251      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5


I0000 00:00:1775312638.981324      68 service.cc:152] XLA service 0x7fe29c08f200 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775312638.981365      68 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775312639.430382      68 cuda_dnn.cc:529] Loaded cuDNN version 91002


  4/528 ━━━━━━━━━━━━━━━━━━━━ 26s 50ms/step - loss: 0.7528 - mae: 0.5377

I0000 00:00:1775312648.753537      68 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


528/528 ━━━━━━━━━━━━━━━━━━━━ 55s 83ms/step - loss: 0.3056 - mae: 0.1829 - val_loss: 0.7201 - val_mae: 0.2549
Epoch 2/5
528/528 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - loss: 0.2590 - mae: 0.1114 - val_loss: 0.7552 - val_mae: 0.2817
Epoch 3/5
528/528 ━━━━━━━━━━━━━━━━━━━━ 38s 73ms/step - loss: 0.2153 - mae: 0.1046 - val_loss: 0.7847 - val_mae: 0.2791
Epoch 4/5
528/528 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - loss: 0.1792 - mae: 0.1009 - val_loss: 0.7791 - val_mae: 0.2554
Epoch 5/5
528/528 ━━━━━━━━━━━━━━━━━━━━ 41s 77ms/step - loss: 0.1732 - mae: 0.0995 - val_loss: 0.7547 - val_mae: 0.2709


In [2]:
import numpy as np
import os

print("Starting Final Inference...")

# 1. Define the Test Input Path
test_in = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/"

# 2. Load Test Features (Only the 4 we trained on)
print("Loading test features...")
t_c = np.load(os.path.join(test_in, "cpm25.npy")).astype(np.float32)
t_u = np.load(os.path.join(test_in, "u10.npy")).astype(np.float32)
t_v = np.load(os.path.join(test_in, "v10.npy")).astype(np.float32)
t_p = np.load(os.path.join(test_in, "pblh.npy")).astype(np.float32)

# 3. Stack and Standardize (Using the exact same 'stats' from training)
# Shape: (Samples, 10, 140, 124, 4)
X_test_raw = np.stack([t_c, t_u, t_v, t_p], axis=-1)
X_test_std = (X_test_raw - stats[0]) / (stats[1] + 1e-7)

# 4. Flatten Time into Channels for the 2D CNN
# (Samples, 10, 140, 124, 4) -> (Samples, 140, 124, 40)
num_samples = X_test_std.shape[0]
X_test_input = np.transpose(X_test_std, (0, 2, 3, 1, 4)).reshape(num_samples, 140, 124, 40)

# 5. Predict the Standardized Deltas
print("Predicting deltas...")
pred_deltas_std = model.predict(X_test_input, batch_size=4) 

# 6. Reconstruct Absolute PM2.5 Values
# A. Get the last known standardized PM2.5 (Hour 10)
last_known_std = X_test_std[:, 9, :, :, 0] # Shape: (Samples, 140, 124)

# B. Add predicted delta to the last known state
# We expand dims so the (Samples, 140, 124) adds to all 16 future channels
pred_abs_std = np.expand_dims(last_known_std, axis=-1) + pred_deltas_std

# C. Reverse Z-Score: (Value * Std) + Mean
# Use index 0 because PM2.5 is the first channel
final_preds_spatial = (pred_abs_std * stats[1][0]) + stats[0][0]

# Current shape: (218, 16, 140, 124)
# We want to move axis 1 (16) to the end.
# New order: axis 0 (218), axis 2 (140), axis 3 (124), axis 1 (16)

final_preds_fixed = np.transpose(final_preds, (0, 2, 3, 1))

print(f"Corrected shape: {final_preds_fixed.shape}") 
# Should print: (218, 140, 124, 16)

# Save the corrected version
np.save('/kaggle/working/preds.npy', final_preds_fixed.astype(np.float32))

Starting Final Inference...
Loading test features...
Predicting deltas...
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step


NameError: name 'final_preds' is not defined

In [ ]:
from IPython.display import FileLink
FileLink(r'preds.npy')